# Decision Trees: A Comprehensive Guide

This notebook provides an in-depth exploration of Decision Trees for classification, covering theory, implementation from scratch, practical usage with scikit-learn, and best practices.

**Table of Contents:**
1. [Theory Section](#1.-Theory-Section)
2. [Implementation from Scratch](#2.-Implementation-from-Scratch)
3. [Training & Optimization](#3.-Training-&-Optimization)
4. [Diagnostics & Evaluation](#4.-Diagnostics-&-Evaluation)
5. [Visualizations](#5.-Visualizations)
6. [Use Cases & Guidelines](#6.-Use-Cases-&-Guidelines)
7. [Comparison with sklearn](#7.-Comparison-with-sklearn)

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier as SklearnDT
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")

---
## 1. Theory Section

### 1.1 What is a Decision Tree?

A Decision Tree is a supervised learning algorithm that creates a model which predicts the value of a target variable by learning simple decision rules inferred from data features. The tree structure consists of:

- **Root Node**: The topmost node representing the entire dataset
- **Internal Nodes**: Decision points that split data based on feature values
- **Leaf Nodes**: Terminal nodes that provide the final classification
- **Branches**: Connections between nodes representing decision outcomes

### 1.2 Splitting Criteria

#### 1.2.1 Entropy

Entropy measures the impurity or uncertainty in a dataset. For a dataset $S$ with $K$ classes:

$$H(S) = -\sum_{k=1}^{K} p_k \log_2(p_k)$$

where $p_k$ is the proportion of samples belonging to class $k$.

- **Entropy = 0**: All samples belong to one class (pure node)
- **Entropy = 1** (for binary): Equal distribution between classes (maximum impurity)

#### 1.2.2 Information Gain

Information Gain measures the reduction in entropy after splitting on a feature $A$:

$$IG(S, A) = H(S) - \sum_{v \in Values(A)} \frac{|S_v|}{|S|} H(S_v)$$

where:
- $H(S)$ is the entropy of the original set
- $S_v$ is the subset of $S$ for which feature $A$ has value $v$
- The summation computes the weighted average entropy after splitting

**Higher Information Gain = Better Split**

#### 1.2.3 Gini Impurity

Gini Impurity measures the probability of incorrectly classifying a randomly chosen element:

$$Gini(S) = 1 - \sum_{k=1}^{K} p_k^2$$

- **Gini = 0**: Pure node (all samples belong to one class)
- **Gini = 0.5** (for binary): Maximum impurity (equal distribution)

**Lower Gini after split = Better Split**

### 1.3 Tree Building Algorithms

#### 1.3.1 ID3 (Iterative Dichotomiser 3)
- Uses **Information Gain** as splitting criterion
- Works only with **categorical features**
- Creates multi-way splits (one branch per category)
- Does not handle missing values natively

#### 1.3.2 C4.5 (Successor of ID3)
- Uses **Gain Ratio** to address Information Gain's bias toward features with many values:
$$GainRatio(S, A) = \frac{IG(S, A)}{SplitInfo(S, A)}$$
$$SplitInfo(S, A) = -\sum_{v \in Values(A)} \frac{|S_v|}{|S|} \log_2\frac{|S_v|}{|S|}$$
- Handles **continuous features** by finding optimal split points
- Handles **missing values**
- Includes **pruning** to reduce overfitting

#### 1.3.3 CART (Classification and Regression Trees)
- Uses **Gini Impurity** for classification, **MSE** for regression
- Creates **binary splits** only
- Handles both categorical and continuous features
- Used by scikit-learn's DecisionTreeClassifier

### 1.4 Pruning

Pruning reduces tree complexity to prevent overfitting:

#### Pre-pruning (Early Stopping)
- **max_depth**: Limit tree depth
- **min_samples_split**: Minimum samples required to split a node
- **min_samples_leaf**: Minimum samples required in a leaf node
- **max_features**: Limit features considered for splitting

#### Post-pruning
- Build full tree, then remove branches that don't improve validation performance
- **Reduced Error Pruning**: Replace subtree with leaf if it doesn't increase error
- **Cost Complexity Pruning (CCP)**: Balance tree complexity vs. accuracy

### 1.5 Time and Space Complexity

Let $n$ = number of samples, $m$ = number of features, $d$ = tree depth

| Operation | Time Complexity | Space Complexity |
|-----------|----------------|------------------|
| Training  | $O(m \cdot n \cdot \log n)$ to $O(m \cdot n^2)$ | $O(n)$ |
| Prediction | $O(d)$ per sample | $O(d)$ |
| Storage   | - | $O(2^d)$ nodes |

**Notes:**
- Training complexity depends on whether data is pre-sorted
- Balanced tree: $d \approx \log_2(n)$
- Worst case (unbalanced): $d = n$

---
## 2. Implementation from Scratch

We'll implement a `DecisionTreeClassifier` using only NumPy, supporting both Gini and Entropy criteria.

In [ ]:
class Node:
    """
    Represents a node in the decision tree.
    
    Attributes:
        feature_index: Index of feature used for splitting (None for leaf nodes)
        threshold: Threshold value for splitting (None for leaf nodes)
        left: Left child node (samples where feature <= threshold)
        right: Right child node (samples where feature > threshold)
        value: Class label for leaf nodes (None for internal nodes)
        n_samples: Number of samples at this node
        impurity: Impurity value at this node
    """
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, 
                 value=None, n_samples=0, impurity=0.0):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        self.n_samples = n_samples
        self.impurity = impurity
    
    def is_leaf(self):
        """Check if this node is a leaf node."""
        return self.value is not None

In [ ]:
class DecisionTreeClassifier:
    """
    Decision Tree Classifier implemented from scratch using NumPy.
    
    Parameters:
        criterion: str, 'gini' or 'entropy' (default='gini')
            The function to measure the quality of a split.
        max_depth: int or None (default=None)
            Maximum depth of the tree. If None, nodes are expanded until
            all leaves are pure or contain less than min_samples_split samples.
        min_samples_split: int (default=2)
            Minimum number of samples required to split an internal node.
        min_samples_leaf: int (default=1)
            Minimum number of samples required to be at a leaf node.
    
    Attributes:
        root_: Node
            The root node of the fitted tree.
        n_classes_: int
            Number of classes.
        n_features_: int
            Number of features.
        feature_importances_: array of shape (n_features,)
            Feature importances computed as the normalized total reduction
            of the criterion brought by that feature.
    """
    
    def __init__(self, criterion='gini', max_depth=None, min_samples_split=2, min_samples_leaf=1):
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.root_ = None
        self.n_classes_ = None
        self.n_features_ = None
        self.feature_importances_ = None
        self._feature_importance_accumulator = None
    
    def _entropy(self, y):
        """
        Calculate entropy of a label array.
        
        H(S) = -sum(p_k * log2(p_k)) for all classes k
        """
        if len(y) == 0:
            return 0.0
        
        # Count occurrences of each class
        _, counts = np.unique(y, return_counts=True)
        probabilities = counts / len(y)
        
        # Avoid log(0) by filtering out zero probabilities
        probabilities = probabilities[probabilities > 0]
        
        return -np.sum(probabilities * np.log2(probabilities))
    
    def _gini(self, y):
        """
        Calculate Gini impurity of a label array.
        
        Gini(S) = 1 - sum(p_k^2) for all classes k
        """
        if len(y) == 0:
            return 0.0
        
        # Count occurrences of each class
        _, counts = np.unique(y, return_counts=True)
        probabilities = counts / len(y)
        
        return 1 - np.sum(probabilities ** 2)
    
    def _impurity(self, y):
        """Calculate impurity based on the selected criterion."""
        if self.criterion == 'entropy':
            return self._entropy(y)
        else:
            return self._gini(y)
    
    def _information_gain(self, y, y_left, y_right):
        """
        Calculate information gain from a split.
        
        IG = H(parent) - weighted_avg(H(children))
        """
        n = len(y)
        n_left = len(y_left)
        n_right = len(y_right)
        
        if n_left == 0 or n_right == 0:
            return 0.0
        
        parent_impurity = self._impurity(y)
        weighted_child_impurity = (n_left / n) * self._impurity(y_left) + \
                                   (n_right / n) * self._impurity(y_right)
        
        return parent_impurity - weighted_child_impurity
    
    def _best_split(self, X, y):
        """
        Find the best split for a node.
        
        Returns:
            best_feature: Index of the best feature to split on
            best_threshold: Best threshold value for the split
            best_gain: Information gain from the best split
        """
        n_samples, n_features = X.shape
        
        best_gain = -np.inf
        best_feature = None
        best_threshold = None
        
        # Iterate over all features
        for feature_idx in range(n_features):
            feature_values = X[:, feature_idx]
            
            # Get unique thresholds (midpoints between consecutive unique values)
            unique_values = np.unique(feature_values)
            if len(unique_values) == 1:
                continue
            
            # Calculate thresholds as midpoints between consecutive unique values
            thresholds = (unique_values[:-1] + unique_values[1:]) / 2
            
            # Evaluate each threshold
            for threshold in thresholds:
                # Split the data
                left_mask = feature_values <= threshold
                right_mask = ~left_mask
                
                # Check minimum samples constraint
                if np.sum(left_mask) < self.min_samples_leaf or \
                   np.sum(right_mask) < self.min_samples_leaf:
                    continue
                
                y_left = y[left_mask]
                y_right = y[right_mask]
                
                # Calculate information gain
                gain = self._information_gain(y, y_left, y_right)
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature_idx
                    best_threshold = threshold
        
        return best_feature, best_threshold, best_gain
    
    def _build_tree(self, X, y, depth=0):
        """
        Recursively build the decision tree.
        
        Parameters:
            X: Feature matrix
            y: Target labels
            depth: Current depth of the tree
        
        Returns:
            Node: The root node of the (sub)tree
        """
        n_samples = len(y)
        n_classes = len(np.unique(y))
        current_impurity = self._impurity(y)
        
        # Stopping conditions for creating a leaf node
        # 1. Pure node (all samples belong to one class)
        # 2. Maximum depth reached
        # 3. Not enough samples to split
        if (n_classes == 1 or 
            (self.max_depth is not None and depth >= self.max_depth) or
            n_samples < self.min_samples_split):
            
            # Create leaf node with majority class
            leaf_value = self._most_common_class(y)
            return Node(value=leaf_value, n_samples=n_samples, impurity=current_impurity)
        
        # Find the best split
        best_feature, best_threshold, best_gain = self._best_split(X, y)
        
        # If no valid split found, create leaf node
        if best_feature is None or best_gain <= 0:
            leaf_value = self._most_common_class(y)
            return Node(value=leaf_value, n_samples=n_samples, impurity=current_impurity)
        
        # Accumulate feature importance
        # Importance = n_samples * impurity_reduction
        self._feature_importance_accumulator[best_feature] += n_samples * best_gain
        
        # Split the data
        left_mask = X[:, best_feature] <= best_threshold
        right_mask = ~left_mask
        
        # Recursively build left and right subtrees
        left_subtree = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_subtree = self._build_tree(X[right_mask], y[right_mask], depth + 1)
        
        return Node(feature_index=best_feature, threshold=best_threshold,
                    left=left_subtree, right=right_subtree,
                    n_samples=n_samples, impurity=current_impurity)
    
    def _most_common_class(self, y):
        """Return the most common class in y."""
        classes, counts = np.unique(y, return_counts=True)
        return classes[np.argmax(counts)]
    
    def fit(self, X, y):
        """
        Build a decision tree classifier from the training set (X, y).
        
        Parameters:
            X: array-like of shape (n_samples, n_features)
                Training feature matrix.
            y: array-like of shape (n_samples,)
                Target labels.
        
        Returns:
            self: Fitted estimator.
        """
        X = np.array(X)
        y = np.array(y)
        
        self.n_features_ = X.shape[1]
        self.n_classes_ = len(np.unique(y))
        self._feature_importance_accumulator = np.zeros(self.n_features_)
        
        # Build the tree
        self.root_ = self._build_tree(X, y)
        
        # Normalize feature importances
        total_importance = np.sum(self._feature_importance_accumulator)
        if total_importance > 0:
            self.feature_importances_ = self._feature_importance_accumulator / total_importance
        else:
            self.feature_importances_ = np.zeros(self.n_features_)
        
        return self
    
    def _predict_single(self, x, node):
        """
        Predict class for a single sample by traversing the tree.
        
        Parameters:
            x: Single sample feature vector
            node: Current node in the tree
        
        Returns:
            Predicted class label
        """
        # If we reached a leaf node, return its value
        if node.is_leaf():
            return node.value
        
        # Traverse left or right based on the feature value
        if x[node.feature_index] <= node.threshold:
            return self._predict_single(x, node.left)
        else:
            return self._predict_single(x, node.right)
    
    def predict(self, X):
        """
        Predict class labels for samples in X.
        
        Parameters:
            X: array-like of shape (n_samples, n_features)
                Feature matrix.
        
        Returns:
            y_pred: array of shape (n_samples,)
                Predicted class labels.
        """
        X = np.array(X)
        return np.array([self._predict_single(x, self.root_) for x in X])
    
    def score(self, X, y):
        """
        Return the mean accuracy on the given test data and labels.
        
        Parameters:
            X: array-like of shape (n_samples, n_features)
                Test feature matrix.
            y: array-like of shape (n_samples,)
                True labels.
        
        Returns:
            score: float
                Mean accuracy.
        """
        return np.mean(self.predict(X) == y)
    
    def get_depth(self):
        """Return the depth of the decision tree."""
        def _depth(node):
            if node is None or node.is_leaf():
                return 0
            return 1 + max(_depth(node.left), _depth(node.right))
        return _depth(self.root_)
    
    def get_n_leaves(self):
        """Return the number of leaves in the decision tree."""
        def _count_leaves(node):
            if node is None:
                return 0
            if node.is_leaf():
                return 1
            return _count_leaves(node.left) + _count_leaves(node.right)
        return _count_leaves(self.root_)
    
    def print_tree(self, feature_names=None, class_names=None):
        """
        Print a text representation of the decision tree.
        
        Parameters:
            feature_names: list of str, optional
                Names of features.
            class_names: list of str, optional
                Names of classes.
        """
        def _print_node(node, depth=0, prefix="Root"):
            indent = "    " * depth
            
            if node.is_leaf():
                class_label = class_names[int(node.value)] if class_names else str(node.value)
                print(f"{indent}{prefix}: [Leaf] class={class_label}, samples={node.n_samples}")
            else:
                feature = feature_names[node.feature_index] if feature_names else f"X[{node.feature_index}]"
                print(f"{indent}{prefix}: [{feature} <= {node.threshold:.4f}] samples={node.n_samples}, impurity={node.impurity:.4f}")
                _print_node(node.left, depth + 1, "L")
                _print_node(node.right, depth + 1, "R")
        
        if self.root_ is None:
            print("Tree has not been fitted yet.")
        else:
            _print_node(self.root_)

print("DecisionTreeClassifier class defined successfully!")

### 2.1 Testing the Implementation

In [ ]:
# Create a simple test dataset
X_test = np.array([
    [2.5, 3.0],
    [1.0, 1.0],
    [3.0, 1.5],
    [4.0, 4.0],
    [3.5, 3.5],
    [2.0, 2.0],
    [1.5, 3.0],
    [3.0, 2.5]
])
y_test = np.array([0, 0, 0, 1, 1, 0, 0, 1])

# Test with Gini criterion
print("Testing with Gini criterion:")
dt_gini = DecisionTreeClassifier(criterion='gini', max_depth=3)
dt_gini.fit(X_test, y_test)
print(f"Predictions: {dt_gini.predict(X_test)}")
print(f"Accuracy: {dt_gini.score(X_test, y_test):.4f}")
print(f"Tree depth: {dt_gini.get_depth()}")
print(f"Number of leaves: {dt_gini.get_n_leaves()}")
print()

# Test with Entropy criterion
print("Testing with Entropy criterion:")
dt_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=3)
dt_entropy.fit(X_test, y_test)
print(f"Predictions: {dt_entropy.predict(X_test)}")
print(f"Accuracy: {dt_entropy.score(X_test, y_test):.4f}")
print(f"Tree depth: {dt_entropy.get_depth()}")
print(f"Number of leaves: {dt_entropy.get_n_leaves()}")

---
## 3. Training & Optimization

We'll use the Wine dataset from scikit-learn to demonstrate training and hyperparameter optimization.

In [ ]:
# Load the Wine dataset
wine = load_wine()
X = wine.data
y = wine.target
feature_names = wine.feature_names
class_names = wine.target_names

print("Wine Dataset Information:")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Class names: {class_names}")
print(f"\nFeature names:")
for i, name in enumerate(feature_names):
    print(f"  {i}: {name}")

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"\nClass distribution in training set:")
for i, name in enumerate(class_names):
    count = np.sum(y_train == i)
    print(f"  {name}: {count} ({count/len(y_train)*100:.1f}%)")

In [ ]:
# Train our custom Decision Tree with default parameters
print("Training Decision Tree with default parameters...")
dt_custom = DecisionTreeClassifier(criterion='gini')
dt_custom.fit(X_train, y_train)

# Evaluate on training and test sets
train_acc = dt_custom.score(X_train, y_train)
test_acc = dt_custom.score(X_test, y_test)

print(f"\nResults (no regularization):")
print(f"Training accuracy: {train_acc:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Tree depth: {dt_custom.get_depth()}")
print(f"Number of leaves: {dt_custom.get_n_leaves()}")

In [ ]:
# Hyperparameter tuning: Testing different max_depth values
print("Hyperparameter Tuning: max_depth")
print("-" * 60)

depths = [1, 2, 3, 4, 5, 6, 7, None]
results = []

for depth in depths:
    dt = DecisionTreeClassifier(criterion='gini', max_depth=depth, min_samples_split=2)
    dt.fit(X_train, y_train)
    
    train_acc = dt.score(X_train, y_train)
    test_acc = dt.score(X_test, y_test)
    actual_depth = dt.get_depth()
    n_leaves = dt.get_n_leaves()
    
    results.append({
        'max_depth': depth if depth else 'None',
        'actual_depth': actual_depth,
        'n_leaves': n_leaves,
        'train_acc': train_acc,
        'test_acc': test_acc
    })
    
    depth_str = str(depth) if depth else 'None'
    print(f"max_depth={depth_str:4s} | depth={actual_depth} | leaves={n_leaves:3d} | "
          f"train={train_acc:.4f} | test={test_acc:.4f}")

# Find best depth based on test accuracy
best_result = max(results, key=lambda x: x['test_acc'])
print(f"\nBest configuration: max_depth={best_result['max_depth']} with test accuracy={best_result['test_acc']:.4f}")

In [ ]:
# Hyperparameter tuning: Testing min_samples_split
print("Hyperparameter Tuning: min_samples_split")
print("-" * 60)

min_samples_values = [2, 5, 10, 15, 20, 30]

for min_samples in min_samples_values:
    dt = DecisionTreeClassifier(criterion='gini', max_depth=5, min_samples_split=min_samples)
    dt.fit(X_train, y_train)
    
    train_acc = dt.score(X_train, y_train)
    test_acc = dt.score(X_test, y_test)
    actual_depth = dt.get_depth()
    n_leaves = dt.get_n_leaves()
    
    print(f"min_samples_split={min_samples:2d} | depth={actual_depth} | leaves={n_leaves:3d} | "
          f"train={train_acc:.4f} | test={test_acc:.4f}")

In [ ]:
# Train the optimized model
print("Training optimized model...")
dt_optimized = DecisionTreeClassifier(
    criterion='gini',
    max_depth=4,
    min_samples_split=5,
    min_samples_leaf=2
)
dt_optimized.fit(X_train, y_train)

print(f"\nOptimized Model Results:")
print(f"Training accuracy: {dt_optimized.score(X_train, y_train):.4f}")
print(f"Test accuracy: {dt_optimized.score(X_test, y_test):.4f}")
print(f"Tree depth: {dt_optimized.get_depth()}")
print(f"Number of leaves: {dt_optimized.get_n_leaves()}")

---
## 4. Diagnostics & Evaluation

In [ ]:
# Generate predictions
y_pred = dt_optimized.predict(X_test)

# Confusion Matrix
print("Confusion Matrix:")
print("="*50)
cm = confusion_matrix(y_test, y_pred)

# Print confusion matrix with class names
print(f"\n{'':12s}", end="")
for name in class_names:
    print(f"{name[:8]:>10s}", end="")
print("  <- Predicted")
print("-" * 50)

for i, name in enumerate(class_names):
    print(f"{name[:10]:12s}", end="")
    for j in range(len(class_names)):
        print(f"{cm[i,j]:10d}", end="")
    print()
print("Actual |")
print("       v")

In [ ]:
# Classification Report
print("\nClassification Report:")
print("="*60)
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
# Feature Importance
print("\nFeature Importances:")
print("="*60)

# Sort features by importance
importance_indices = np.argsort(dt_optimized.feature_importances_)[::-1]

for i, idx in enumerate(importance_indices):
    importance = dt_optimized.feature_importances_[idx]
    if importance > 0:
        bar = "#" * int(importance * 40)
        print(f"{i+1:2d}. {feature_names[idx]:25s}: {importance:.4f} {bar}")

In [ ]:
# Text-based Tree Visualization
print("\nDecision Tree Structure (Text Visualization):")
print("="*80)
dt_optimized.print_tree(feature_names=feature_names, class_names=list(class_names))

---
## 5. Visualizations

In [ ]:
# Feature Importance Bar Chart
plt.figure(figsize=(12, 6))

# Sort features by importance
importance_indices = np.argsort(dt_optimized.feature_importances_)[::-1]
sorted_importances = dt_optimized.feature_importances_[importance_indices]
sorted_names = [feature_names[i] for i in importance_indices]

# Create bar chart
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(sorted_names)))
bars = plt.barh(range(len(sorted_names)), sorted_importances, color=colors)

# Customize the plot
plt.yticks(range(len(sorted_names)), sorted_names)
plt.xlabel('Feature Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Decision Tree Feature Importances (Wine Dataset)', fontsize=14)
plt.gca().invert_yaxis()  # Highest importance at top

# Add value labels on bars
for i, (bar, importance) in enumerate(zip(bars, sorted_importances)):
    if importance > 0:
        plt.text(importance + 0.01, bar.get_y() + bar.get_height()/2,
                f'{importance:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Decision Boundary Visualization
# We'll use the two most important features for 2D visualization

# Get the two most important features
top_2_features = importance_indices[:2]
X_2d = X[:, top_2_features]

# Train a new tree on just these two features
dt_2d = DecisionTreeClassifier(criterion='gini', max_depth=4, min_samples_split=5)
dt_2d.fit(X_2d, y)

# Create a mesh grid for decision boundary visualization
x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5

# Create mesh grid
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 200),
    np.linspace(y_min, y_max, 200)
)

# Get predictions for each point in the mesh
Z = dt_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Create the plot
fig, ax = plt.subplots(figsize=(12, 8))

# Plot decision boundary
contour = ax.contourf(xx, yy, Z, alpha=0.4, cmap=plt.cm.RdYlBu)
ax.contour(xx, yy, Z, colors='black', linewidths=0.5)

# Plot training points
scatter_colors = ['red', 'green', 'blue']
markers = ['o', 's', '^']

for i, (color, marker, name) in enumerate(zip(scatter_colors, markers, class_names)):
    mask = y == i
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, marker=marker,
               label=name, edgecolors='black', s=60, alpha=0.7)

ax.set_xlabel(feature_names[top_2_features[0]], fontsize=12)
ax.set_ylabel(feature_names[top_2_features[1]], fontsize=12)
ax.set_title('Decision Tree Decision Boundaries\n(Using Top 2 Important Features)', fontsize=14)
ax.legend(loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

print(f"\nNote: Decision boundaries are shown using the two most important features:")
print(f"  - {feature_names[top_2_features[0]]}")
print(f"  - {feature_names[top_2_features[1]]}")

In [ ]:
# Confusion Matrix Heatmap
fig, ax = plt.subplots(figsize=(8, 6))

# Create heatmap
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)

# Set ticks and labels
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=class_names,
       yticklabels=class_names,
       ylabel='True label',
       xlabel='Predicted label',
       title='Confusion Matrix Heatmap')

# Rotate the tick labels and set their alignment
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Loop over data dimensions and create text annotations
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Training vs Test Accuracy for Different Tree Depths
depths = range(1, 12)
train_scores = []
test_scores = []

for depth in depths:
    dt = DecisionTreeClassifier(criterion='gini', max_depth=depth)
    dt.fit(X_train, y_train)
    train_scores.append(dt.score(X_train, y_train))
    test_scores.append(dt.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(depths, train_scores, 'o-', color='blue', label='Training Accuracy', linewidth=2, markersize=8)
ax.plot(depths, test_scores, 's-', color='red', label='Test Accuracy', linewidth=2, markersize=8)

# Mark the best depth
best_depth = depths[np.argmax(test_scores)]
best_test_score = max(test_scores)
ax.axvline(x=best_depth, color='green', linestyle='--', alpha=0.7, label=f'Best depth = {best_depth}')

ax.set_xlabel('Tree Depth', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Training vs Test Accuracy vs Tree Depth\n(Overfitting Analysis)', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xticks(depths)

# Add annotation for overfitting region
ax.annotate('Overfitting region', xy=(8, 0.92), xytext=(8, 0.85),
            fontsize=10, ha='center',
            arrowprops=dict(arrowstyle='->', color='gray'))

plt.tight_layout()
plt.show()

---
## 6. Use Cases & Guidelines

### 6.1 When to Use Decision Trees

**Best Use Cases:**

1. **Interpretability is Critical**
   - Healthcare: Diagnosing diseases where doctors need to understand the reasoning
   - Finance: Credit scoring where regulatory compliance requires explainable decisions
   - Legal: Risk assessment where decisions must be justified

2. **Feature Selection/Importance**
   - Identifying which features matter most in a dataset
   - Initial exploratory data analysis
   - Feature engineering insights

3. **Mixed Data Types**
   - Datasets with both numerical and categorical features
   - No need for extensive preprocessing or feature scaling

4. **Non-linear Relationships**
   - When linear models underperform
   - Data with complex decision boundaries

5. **Quick Baseline Models**
   - Fast training and prediction
   - Good starting point before trying complex models

6. **Ensemble Building Blocks**
   - Base learners for Random Forests, Gradient Boosting, etc.

### 6.2 When NOT to Use Decision Trees

**Avoid Decision Trees When:**

1. **High-Dimensional Sparse Data**
   - Text classification with bag-of-words features
   - One-hot encoded categorical features with many categories
   - Better alternatives: Linear models, Naive Bayes

2. **Extrapolation Required**
   - Time series forecasting beyond training range
   - Predicting values outside the training distribution
   - Trees can only predict values seen during training

3. **Smooth Decision Boundaries Needed**
   - When boundaries are naturally smooth/curved
   - Trees create axis-parallel, step-like boundaries
   - Better alternatives: SVM with RBF kernel, Neural Networks

4. **Very Large Datasets**
   - Memory constraints with millions of samples
   - Training time becomes prohibitive
   - Consider: Sampling, online learning methods

5. **Class Imbalance Without Handling**
   - Trees tend to favor majority class
   - Requires careful class weighting or resampling

### 6.3 Pros and Cons

| Pros | Cons |
|------|------|
| Easy to understand and interpret | Prone to overfitting |
| Requires little data preprocessing | Unstable (small data changes cause different trees) |
| Handles both numerical and categorical data | Biased toward features with more levels |
| Can capture non-linear relationships | Cannot extrapolate |
| Fast prediction (O(log n) depth) | Greedy algorithm may not find global optimum |
| Feature importance built-in | Axis-parallel splits only |
| Works with missing values (some implementations) | Can create biased trees if classes imbalanced |

### 6.4 Overfitting Considerations

**Signs of Overfitting:**
- Training accuracy >> Test accuracy
- Deep trees with many leaves
- High variance across different random states

**Prevention Strategies:**

1. **Pre-pruning (Regularization)**
   ```python
   # Limit tree growth
   max_depth = 5           # Limit tree depth
   min_samples_split = 10  # Minimum samples to split
   min_samples_leaf = 5    # Minimum samples in leaf
   max_features = 'sqrt'   # Consider subset of features
   ```

2. **Post-pruning**
   ```python
   # sklearn's cost complexity pruning
   ccp_alpha = 0.01  # Complexity parameter
   ```

3. **Cross-Validation**
   - Use k-fold CV to tune hyperparameters
   - Monitor both training and validation performance

4. **Ensemble Methods**
   - Random Forest: Reduces variance through bagging
   - Gradient Boosting: Reduces bias incrementally

**Rule of Thumb:**
- Start with a shallow tree (depth 3-5)
- Gradually increase depth while monitoring test accuracy
- Stop when test accuracy starts decreasing

---
## 7. Comparison with sklearn

In [ ]:
# Train sklearn's DecisionTreeClassifier with the same parameters
sklearn_dt = SklearnDT(
    criterion='gini',
    max_depth=4,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
sklearn_dt.fit(X_train, y_train)

# Our custom implementation
custom_dt = DecisionTreeClassifier(
    criterion='gini',
    max_depth=4,
    min_samples_split=5,
    min_samples_leaf=2
)
custom_dt.fit(X_train, y_train)

print("Model Comparison: Custom vs sklearn")
print("=" * 60)

In [ ]:
# Compare accuracy
print("\nAccuracy Comparison:")
print("-" * 40)
print(f"{'Metric':<20} {'Custom':>12} {'sklearn':>12}")
print("-" * 40)

custom_train = custom_dt.score(X_train, y_train)
custom_test = custom_dt.score(X_test, y_test)
sklearn_train = sklearn_dt.score(X_train, y_train)
sklearn_test = sklearn_dt.score(X_test, y_test)

print(f"{'Training Accuracy':<20} {custom_train:>12.4f} {sklearn_train:>12.4f}")
print(f"{'Test Accuracy':<20} {custom_test:>12.4f} {sklearn_test:>12.4f}")

In [ ]:
# Compare tree structure
print("\nTree Structure Comparison:")
print("-" * 40)
print(f"{'Property':<20} {'Custom':>12} {'sklearn':>12}")
print("-" * 40)

print(f"{'Tree Depth':<20} {custom_dt.get_depth():>12} {sklearn_dt.get_depth():>12}")
print(f"{'Number of Leaves':<20} {custom_dt.get_n_leaves():>12} {sklearn_dt.get_n_leaves():>12}")

In [ ]:
# Compare feature importances
print("\nFeature Importance Comparison:")
print("-" * 60)
print(f"{'Feature':<25} {'Custom':>12} {'sklearn':>12} {'Diff':>10}")
print("-" * 60)

for i, name in enumerate(feature_names):
    custom_imp = custom_dt.feature_importances_[i]
    sklearn_imp = sklearn_dt.feature_importances_[i]
    diff = abs(custom_imp - sklearn_imp)
    if custom_imp > 0 or sklearn_imp > 0:
        print(f"{name:<25} {custom_imp:>12.4f} {sklearn_imp:>12.4f} {diff:>10.4f}")

In [ ]:
# Compare prediction time
import time

# Warm up
_ = custom_dt.predict(X_test)
_ = sklearn_dt.predict(X_test)

# Time custom implementation
n_iterations = 100

start = time.time()
for _ in range(n_iterations):
    _ = custom_dt.predict(X_test)
custom_time = (time.time() - start) / n_iterations * 1000  # ms

start = time.time()
for _ in range(n_iterations):
    _ = sklearn_dt.predict(X_test)
sklearn_time = (time.time() - start) / n_iterations * 1000  # ms

print("\nPrediction Time Comparison (per batch):")
print("-" * 40)
print(f"Custom implementation: {custom_time:.4f} ms")
print(f"sklearn implementation: {sklearn_time:.4f} ms")
print(f"Ratio (custom/sklearn): {custom_time/sklearn_time:.2f}x")

In [ ]:
# Compare predictions
custom_pred = custom_dt.predict(X_test)
sklearn_pred = sklearn_dt.predict(X_test)

agreement = np.mean(custom_pred == sklearn_pred)

print(f"\nPrediction Agreement:")
print("-" * 40)
print(f"Agreement between custom and sklearn: {agreement:.2%}")
print(f"Number of different predictions: {np.sum(custom_pred != sklearn_pred)} out of {len(X_test)}")

In [ ]:
# Visual comparison of feature importances
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(feature_names))
width = 0.35

bars1 = ax.bar(x - width/2, custom_dt.feature_importances_, width, label='Custom Implementation', color='steelblue')
bars2 = ax.bar(x + width/2, sklearn_dt.feature_importances_, width, label='sklearn', color='coral')

ax.set_xlabel('Feature', fontsize=12)
ax.set_ylabel('Importance', fontsize=12)
ax.set_title('Feature Importance Comparison: Custom vs sklearn', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(feature_names, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print("""
Key Observations:

1. ACCURACY: Both implementations achieve similar accuracy on the Wine
   dataset, demonstrating that our from-scratch implementation correctly
   implements the decision tree algorithm.

2. FEATURE IMPORTANCE: The feature importances may differ slightly due to:
   - Tie-breaking strategies when multiple features have equal information gain
   - Threshold selection methods for continuous features
   - Implementation details in impurity calculation

3. PERFORMANCE: sklearn's implementation is optimized in Cython and uses
   efficient data structures, making it significantly faster than our
   pure Python/NumPy implementation.

4. RECOMMENDATIONS:
   - Use sklearn for production applications (speed, reliability)
   - Use our implementation for learning and understanding the algorithm
   - Both are suitable for small to medium datasets

Key Hyperparameters for Decision Trees:
   - max_depth: Primary regularization parameter
   - min_samples_split: Prevents splitting small nodes
   - min_samples_leaf: Ensures leaves have enough samples
   - criterion: 'gini' (default) vs 'entropy' (often similar results)
""")

---
## Conclusion

This notebook covered:

1. **Theory**: Mathematical foundations including Entropy, Information Gain, and Gini Impurity; tree-building algorithms (ID3, C4.5, CART); pruning techniques; and complexity analysis.

2. **Implementation**: A complete Decision Tree Classifier from scratch using NumPy with both Gini and Entropy criteria, supporting max_depth and min_samples_split hyperparameters.

3. **Training & Optimization**: Applied to the Wine dataset with hyperparameter tuning.

4. **Evaluation**: Confusion matrices, classification reports, and feature importance analysis.

5. **Visualization**: Decision boundaries, feature importance charts, and overfitting analysis.

6. **Best Practices**: When to use decision trees, when to avoid them, and how to prevent overfitting.

7. **sklearn Comparison**: Validated our implementation against scikit-learn's DecisionTreeClassifier.

**Next Steps:**
- Explore ensemble methods (Random Forest, Gradient Boosting) that build upon decision trees
- Implement cost-complexity pruning for post-pruning
- Add support for categorical features without encoding
- Implement regression trees